In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import math
import os
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight

from disciple.pipelines import SamplingType
from evaluations.analysis_utils import (
    load_raw_data,
    process_dataframe,
    compute_pass_at_k,
    sample_examples,
    generate_collie_table,
)
from evaluations.coherency import CoherencyEvaluator
from evaluations.dataset import load_dataset
from evaluations.plotting import (
    plot_breakdown,
    plot_confusion_matrices,
    plot_pass_at_k,
    plot_error_distribution,
    plot_coherency,
    build_method_palette,
    PIPELINE_PALETTE,
)

In [ ]:
%config InlineBackend.figure_format = "retina"
sns.set_theme(context='talk', style='ticks', palette='muted')

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# DataFrame pre-processing

In [ ]:
DATASET = "collie"
SUBTASKS = "Sentence-Level Tasks"
# SUBTASKS = "Paragraph-Level Tasks"

# DATASET = "puzzle"
# SUBTASKS = "all"

EXPERIMENTS = {
    "collie": {
        "Sentence-Level Tasks": {
            "2025-03-13-17-03-40": None,  # vllm baselines
            "2025-03-11-17-52-29": None,  # openai baselines
            # "2025-03-12-12-53-30": None,  # disciple
            "2025-03-18-17-56-14": None,  # disciple w/ step granularity prompt
            "2025-03-23-15-12-02": None,  # disciple oracle
            "2025-05-28-12-24-56": None,  # Llama-3.2-3B
            "2025-05-28-13-04-37": None,  # Llama-3.1-8B
            "2025-05-29-13-59-45": None,  # Qwen-3-1.7B
            "2025-05-30-16-33-09": ["gpt-4o-cot"],  # GPT-4o-cot
        },
        "Paragraph-Level Tasks": {
            "2025-03-13-17-05-26": None,  # vllm baselines
            "2025-03-12-14-20-48": None,  # baselines
            # "2025-03-12-14-16-31": None,  # disciple
            "2025-03-19-21-23-37": None,  # disciple step granularity prompt
            "2025-03-23-15-15-13": None,  # disciple oracle
            "2025-05-27-21-56-19": None,  # Llama-3.2-3B
            "2025-05-28-13-08-15": None,  # Llama-3.1-8B
            "2025-05-28-23-08-29": None,  # Qwen-3-1.7B
            "2025-05-30-16-33-49": ["gpt-4o-cot"],  # GPT-4o-cot
        },
    },
    "puzzle": {
        "all": {
            # "2025-03-19-17-09-34": None,  # puzzles combined (elephant poem)
            # "2025-03-21-16-36-48": None,  # puzzles combined (grant proposal)
            "2025-03-23-00-33-59": None,  # puzzles combined (simplified grant proposal)
            "2025-03-24-18-22-25": None,  # puzzles disciple oracle (simplified grant proposal)
            "2025-07-19-21-08-52": ["gpt-4o-cot"],  # GPT-4o-cot
        }
    },
    "ifeval": {
        "all": {
            "2025-03-20-16-13-50": None,
        }
    },
}

TIMESTAMP_TO_PIPELINES = EXPERIMENTS[DATASET][SUBTASKS]

SAVE_FIGS = False
FIGS_PATH = (
    os.path.join("figures", DATASET, SUBTASKS)
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
)
if SAVE_FIGS:
    os.makedirs(FIGS_PATH, exist_ok=True)

In [ ]:
df_raw = load_raw_data(TIMESTAMP_TO_PIPELINES)
df, df_all_attempts = process_dataframe(
    df_raw,
    dataset=DATASET,
    # rescore=False
    rescore=True if DATASET == "puzzle" else False,
)

# Data validation

In [ ]:
print("=== Dataset Summary ===")
TASK_TYPES = df["task_type"].unique().tolist()
N_TASKS = df["task_id"].nunique()
print(f"Total tasks: {N_TASKS}")
print(df.groupby("task_type")["task_id"].nunique())
print()

print("=== Pipeline Summary ===")
METHOD_NAMES = df["method"].unique().tolist()
SAMPLING_TYPE = dict(set(zip(df["method"], df["sampling_type"])))
N_VALUES = sorted(df["N"].unique().tolist())
assert not df["N"].isnull().any()
print(f"Unique values of N: {N_VALUES}")

def validate_pipeline_counts(df):
    method_counts = df["method"].value_counts()
    method_counts = pd.DataFrame(method_counts).join(
        pd.DataFrame.from_dict(SAMPLING_TYPE, orient="index", columns=["sampling_type"])
    )
    print(method_counts)
    for method_name, data in method_counts.iterrows():
        sampling_type = data["sampling_type"]
        count = data["count"]
        if sampling_type == SamplingType.ADAPTIVE:
            assert count == N_TASKS * sum(N_VALUES)
        elif sampling_type == SamplingType.INDEPENDENT:
            assert count == N_TASKS * max(N_VALUES)
        elif sampling_type == SamplingType.SINGLE_SHOT:
            assert count == N_TASKS
        else:
            raise ValueError(f"Invalid sampling type: {sampling_type}")
validate_pipeline_counts(df)

In [ ]:
# Compute class weights for task types
task_id_to_task_type = {}
for task_id in sorted(df["task_id"].unique()):
    task_type = df[df["task_id"] == task_id].iloc[0]["task_type"]
    task_id_to_task_type[task_id] = task_type
print(Counter(task_id_to_task_type.values()))

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["task_type"]),
    y=list(task_id_to_task_type.values()),
)
class_weights = {
    task_type: weight
    for task_type, weight in zip(np.unique(df["task_type"]), class_weights)
}
df["class_weight"] = df["task_type"].map(class_weights)
print(class_weights)

In [ ]:
PALETTE = build_method_palette(df)
print(PALETTE)

# Analysis of core results
Results timestamped 2025-04-01 and earlier

In [ ]:
# Core results are based on a timestamp cutoff from the original submission
# df_core = df[df["core_result"]].copy()
df_core = df.copy()


df_core["method"] = pd.Categorical(
    df_core["method"],
    categories=[
        m
        for m in df["method"].cat.categories.tolist()
        if m in df_core["method"].unique()
    ],
    ordered=True,
)

df_all_attempts_core = df_all_attempts[df_all_attempts["core_result"]].copy()
df_all_attempts_core["method"] = pd.Categorical(
    df_all_attempts_core["method"],
    categories=df_core["method"].cat.categories.tolist(),
    ordered=True,
)

## High-level analysis

In [ ]:
plot_breakdown(df_core);

In [ ]:
# Coarse-grained validity plot (averages over all N values)
g = sns.catplot(
    kind="bar",
    col="task_type",
    data=df_core,
    x="method",
    y="valid_true",
    hue="method",
    palette=PALETTE,
)
g.set_xticklabels(rotation=90, fontsize=10)

In [ ]:
fig = plot_confusion_matrices(df_core);
if SAVE_FIGS:
    fig.savefig(os.path.join(FIGS_PATH, "confusion_matrices.pdf"), bbox_inches="tight")

## Pass@K analysis

In [ ]:
df_pass_at_k = compute_pass_at_k(df_core, k_values=N_VALUES, n_values=N_VALUES)
df_pass_at_1 = df_pass_at_k[df_pass_at_k["k"] == 1]

df_pass_at_k

In [ ]:
from evaluations.plotting import plot_pass_at_k

fig = plot_pass_at_k(
    df_pass_at_1,
    DATASET,
    SUBTASKS,
    y_label="Pass@1",
    # dash_methods=["o1", "GPT-4o", "GPT-4o-mini"],
    dash_methods=["DisCIPL*-SMC", "DisCIPL*-IS", "DisCIPL*-RS"],
    use_class_weights=True,
    palette=PALETTE,
)

if SAVE_FIGS:
    fig.savefig(
        os.path.join(FIGS_PATH, "pass_at_1.pdf"), bbox_inches="tight", dpi=300
    )

In [ ]:
_df = df_pass_at_1[
        (df_pass_at_1["N"] == df_pass_at_1["N"].max())
        & ~(df_pass_at_1["pipeline_name"].str.contains("oracle"))
    ]
_df["method"] = _df["method"].astype(str)

g = sns.barplot(
    data=_df,
    x="method",
    weights="class_weight",
    y="pass@k",
    hue="method",
    palette=PALETTE,
    capsize=0.1,
    err_kws={"linewidth": 1.5},
)
sns.despine()
plt.xticks(rotation=90, fontsize=10)

methods = _df["method"].unique().tolist()
for patch, method in zip(g.patches, methods):
    # Existing results
    if method in [
        "DisCIPL-SMC (Llama-3.2-1B)",
        "DisCIPL-IS (Llama-3.2-1B)",
        "DisCIPL-RS (Llama-3.2-1B)",
        "Llama-3.2-1B",
        "Llama-3.2-1B (Beam)",
        "GPT-4o-mini",
        "GPT-4o",
        "o1",
    ]:
        patch.set_alpha(0.25)

plt.title(f"{DATASET.upper()} {SUBTASKS} (New Results)")
plt.xlabel("")
plt.ylabel("Pass@1");

In [ ]:
plot_pass_at_k(
    df_pass_at_k[df_pass_at_k["N"] == df_pass_at_k["k"]],
    DATASET,
    SUBTASKS,
    y_label="Pass@k",
    dash_methods=["o1", "GPT-4o", "GPT-4o-mini"],
    use_class_weights=True,
    palette=PALETTE,
)

if SAVE_FIGS:
    fig.savefig(os.path.join(FIGS_PATH, "pass_at_k.pdf"), bbox_inches="tight", dpi=300)

## Synthesis error analysis

In [ ]:
ERROR_ANALYSIS_PIPELINE = "disciple_smc"
# ERROR_ANALYSIS_PIPELINE = "disciple_oracle_smc"

In [ ]:
df_core.groupby(["method"], observed=False)["error_true"].value_counts(normalize=True).to_frame()

In [ ]:
g = plot_error_distribution(df_all_attempts_core, pipeline_name=ERROR_ANALYSIS_PIPELINE)

if SAVE_FIGS:
    g.savefig(
        os.path.join(FIGS_PATH, "error_distribution.pdf"), bbox_inches="tight", dpi=300
    )

In [ ]:
sns.color_palette("tab20")

In [ ]:
g = plot_breakdown(
    df_all_attempts_core[df_all_attempts_core["pipeline_name"] == ERROR_ANALYSIS_PIPELINE],
    x="attempt",
    col=None,
)

g._legend.set_title("")
g._legend.set_bbox_to_anchor((0.9, 0.5))
plt.xlabel("Retry #")

if SAVE_FIGS:
    g.savefig(
        os.path.join(FIGS_PATH, "validity_over_attempts.pdf"),
        bbox_inches="tight",
        dpi=300,
    )

## Text analysis

In [ ]:
df_core.groupby("method", observed=False)["text_length"].describe()

In [ ]:
sns.displot(data=df_core, x="text_length", hue="method", kind="kde", common_norm=False, palette=PALETTE)

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    # display(df.groupby(["method", "task_type", "N"])[["valid_true", "error_true"]].mean())
    display(
        df_pass_at_1[~df_pass_at_1["error_true"]]
        # df_pass_at_1
        .groupby(["method", "task_type", "N"])[["pass@k", "error_true"]]
        .mean()
    )

In [ ]:
df[df["task_type"] == "sent_04"].groupby("N")[["valid_true", "error_true"]].mean()

In [ ]:
N_TASKS_PER_TYPE = 1
RANDOM_STATE = 222
SELECTED_TASK_IDS = (
    df.groupby("task_type").sample(n=N_TASKS_PER_TYPE, random_state=RANDOM_STATE)["task_id"].tolist()
)

# SELECTED_TASK_IDS = (
#     df[df["task_type"] == "sent_04"]["task_id"]
#     .sample(n=1, random_state=RANDOM_STATE)
#     .tolist()
# )

# print(SELECTED_TASK_IDS)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(
        df[df["task_id"].isin(SELECTED_TASK_IDS) & (df["N"] == df["N"].max())]
        .sort_values(
            ["task_type", "task_id", "method", "weight"],
            ascending=[True, True, True, False],
        )[
            [
                "task_type",
                "task_id",
                "method",
                "task_prompt",
                "text",
                "text_length",
                "weight",
                "valid_true",
                "check_result",
            ]
        ]
        .groupby(["task_type", "task_id", "method"]).head(1)
    )

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(
        df[
               (df["pipeline_name"] == "disciple_smc") \
               & (df["N"] == df["N"].max())
        ]
        .sort_values("weight", ascending=False)
        .head(1)
        [["task_id", "method", "task_prompt", "text", "weight", "valid_true", "check_result"]]
    )

In [ ]:
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    display(df[df["weight"] > 0])

## Coherency

In [ ]:
df_samples = sample_examples(df_core, use_check=True, seed=123)
# df_samples = sample_examples(df, use_check=True, seed=456)

# Check that the number of samples is consistent across methods and equal to the number of tasks
n_samples_per_method = df_samples.groupby("method", observed=True).size()
# print(n_samples_per_method)
assert n_samples_per_method.nunique() == 1
assert n_samples_per_method.unique().item() == N_TASKS

In [ ]:
# text_list = df["text"].sample(n=10, random_state=123).tolist()
text_list = df_samples["text"].tolist()
print(len(text_list), len(set(text_list)))

In [ ]:
# coherency = CoherencyEvaluator()
coherency = CoherencyEvaluator(cache_filename="coherency-2025-06-01.json")
unrated_texts = [text for text in (set(text_list)) if text not in coherency.cache]
print(f"Number of unrated texts: {len(unrated_texts)}")

# # coherency.clear_cache()
results = await coherency(text_list, batch_size=50)

df_samples["coherency_score"] = [result["score"] for result in results]
df_samples["coherency_analysis"] = [result["analysis"] for result in results]
df_samples["coherency_cost"] = [
    CoherencyEvaluator.compute_cost(result["completion"]) for result in results
]

In [ ]:
df_samples.groupby("method", observed=True)["coherency_score"].describe()

### Coherency by method and task

In [ ]:
_df = df_samples[
    df_samples["pipeline_name"].isin(
        ["disciple_smc", "disciple_importance", "disciple_rejection", "gpt-4o", "o1"]
    )
]

plot_coherency(_df, show_y_axis=False, rescale_y_axis=False, kind="hist", legend=False)

if SAVE_FIGS:
    plt.savefig(
        os.path.join(FIGS_PATH, "coherency.pdf"), bbox_inches="tight", dpi=300
    )

In [ ]:
_df

### Coherency by method

In [ ]:
sns.displot(
    data=df_samples,
    x="coherency_score",
    hue="method",
    palette=PALETTE,
    kind="kde",
    clip=(0, 10),
    common_norm=True,
    fill=True,
)

In [ ]:
sns.displot(
    data=df_samples,
    x="coherency_score",
    col="task_type",
    hue="method",
    palette=PALETTE,
    kind="kde",
    clip=(0, 10),
    common_norm=True,
    fill=True,
)

In [ ]:
pipeline_names = ["disciple_smc", "disciple_importance"]
# palette = {
#     PIPELINES[k]: PALETTE[PIPELINES[k]]
#     for k in pipeline_names
# }
sns.displot(
    data=df_samples[df_samples["pipeline_name"].isin(pipeline_names)],
    x="coherency_score",
    col="task_type",
    hue="method",
    palette=PALETTE,
    kind="kde",
    clip=(0, 10),
    common_norm=False,
    fill=True,
)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxenplot(
    data=df_samples, x="task_type", y="coherency_score", hue="method", palette=PALETTE
)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

### Coherency by method (valid/invalid)

In [ ]:
sns.displot(
    data=df_samples,
    x="coherency_score",
    col="valid_true",
    hue="method",
    palette=PALETTE,
    kind="kde",
    # kind="hist",
    common_norm=True,
    # stat="proportion",
    # multiple="dodge"
    clip=(0, 10),
    fill=True,
)

In [ ]:
g = sns.displot(
    data=df_samples,
    x="coherency_score",
    col="method",
    hue="valid_true",
    palette={True: "tab:green", False: "tab:red"},
    kind="kde",
    common_norm=True,
    fill=True,
)
g.set(xlim=(0, 10))

In [ ]:
with sns.plotting_context("talk", font_scale=1.5):
    g = sns.displot(
        data=df_samples,
        x="coherency_score",
        row="task_type",
        col="method",
        hue="valid_true",
        palette={True: "tab:green", False: "tab:red"},
        kind="hist",
        stat="frequency",
        common_norm=False,
        fill=True,
        aspect=3.0,
        height=2.0,
        facet_kws={"sharey": None, "xlim": (0, 10), "margin_titles": True},
    )
    g.set_titles(col_template="{col_name}", row_template="{row_name}")

    # for ax in g.axes.flat:
    #     ax.set_yticklabels([])
    #     ax.set_yticks([])
    #     ax.set_ylabel("Freq.")

    g._legend.set_bbox_to_anchor((1.05, 0.5))
    plt.tight_layout()

In [ ]:
with sns.plotting_context("talk", font_scale=1.5):
    g = sns.displot(
        data=df_samples,
        x="coherency_score",
        row="task_type",
        col="method",
        hue="valid_true",
        palette={True: "tab:green", False: "tab:red"},
        kind="kde",
        common_norm=True,
        fill=True,
        aspect=3.0,
        height=2.0,
        facet_kws={"sharey": "row", "xlim": (0, 10), "margin_titles": True},
    )
    g.set_titles(col_template="{col_name}", row_template="{row_name}")

    # for ax in g.axes.flat:
    #     ax.set_yticklabels([])

    g._legend.set_bbox_to_anchor((1.05, 0.5))
    plt.tight_layout()

In [ ]:
g = sns.scatterplot(
    data=df_samples.groupby(["method", "task_type"], observed=True)[["valid_true", "coherency_score"]].mean(),
    x="valid_true",
    y="coherency_score",
    hue="method",
    # style="task_type",
    palette=PALETTE,
)
g.legend_.set_bbox_to_anchor((1, 1))

In [ ]:
g = sns.kdeplot(
    data=df_samples.groupby(["method", "task_type"], observed=True)[["valid_true", "coherency_score"]].mean(),
    x="valid_true",
    y="coherency_score",
    hue="method",
    # style="task_type",
    palette=PALETTE,
    fill=False,
    alpha=0.5,
    clip=((0, 1), (0, 10)),
)
g.legend_.set_bbox_to_anchor((2, 1))

In [ ]:
df_samples.groupby("method", observed=True)["coherency_score"].describe()

### Coherency vs. N

In [ ]:
# df_samples_by_n = []
# for n in N_VALUES:
#     _df = sample_examples(
#         df, N=n, use_check=True, seed=123
#     )
#     # _df = sample_examples(df[df["pipeline_name"].str.contains("disciple")], N=n, use_check=True, seed=123)
#     _df["N"] = n
#     df_samples_by_n.append(_df)

# df_samples_by_n = pd.concat(df_samples_by_n, ignore_index=True)

# # Check that the number of samples is consistent across methods and equal to the number of tasks
# n_samples_per_method = df_samples_by_n.groupby(["method", "N"], observed=True).size()
# print(n_samples_per_method)
# assert n_samples_per_method.nunique() == 1
# assert n_samples_per_method.unique().item() == N_TASKS

In [ ]:
coherency = CoherencyEvaluator()
text_list = df_samples_by_n["text"].tolist()
print(
    "Total texts:", len(text_list),
    "Unique texts:", len(set(text_list)),
    "New texts:", len(set(text_list) - set(coherency.cache.keys())),
)

# coherency.clear_cache()
results = await coherency(text_list, batch_size=20)

df_samples_by_n["coherency_score"] = [result["score"] for result in results]
df_samples_by_n["coherency_analysis"] = [result["analysis"] for result in results]
df_samples_by_n["coherency_cost"] = [
    CoherencyEvaluator.compute_cost(result["completion"]) for result in results
]

In [ ]:
df_samples_by_n["valid_coherency"] = df_samples_by_n["valid_true"] * df_samples_by_n["coherency_score"]

_df = df_samples_by_n[
    df_samples_by_n["pipeline_name"].isin(["disciple_smc", "disciple_importance"])
].copy()
# make method non-categorical for plotting
_df["method"] = _df["method"].astype(str)

g = sns.lineplot(
    data=_df,
    x="N",
    y="coherency_score",
    # y="valid_coherency",
    weights="class_weight",
    # col="task_type",
    hue="method",
    palette=PALETTE,
    # kind="line",
    # errorbar=None,
    linewidth=4,
)
g.set(xticks=N_VALUES, xticklabels=N_VALUES)
# for ax in g.axes.flat:
#     ax.set_title(ax.get_title().split(" = ")[-1])
plt.xlabel("N Particles")
plt.ylabel("Coherency Score")

sns.despine()

g.legend_.set_title("")

if SAVE_FIGS:
    plt.savefig(
        os.path.join(FIGS_PATH, "coherency_by_n.pdf"), bbox_inches="tight", dpi=300
    )

In [ ]:
g = sns.scatterplot(
    data=df_samples_by_n.groupby(["method", "task_type", "N"], observed=True)[
        ["valid_true", "coherency_score", "N"]
    ].mean(),
    x="valid_true",
    y="coherency_score",
    hue="method",
    style="task_type",
    size="N",
    palette=PALETTE,
)
g.legend_.set_bbox_to_anchor((1, 1))

### Text samples

In [ ]:
df_sorted = df_samples.sort_values(by="coherency_score", ascending=False)
# for method in df_sorted["method"].unique():
for method in ["GPT-4o"]:
    print(f"Method: {method}")
    with pd.option_context("display.max_colwidth", None):
        display(
            df_sorted[df_sorted["method"] == method][
                [
                    "coherency_score",
                    "task_type",
                    "task_id",
                    "task_prompt",
                    "valid_true",
                    "check_result",
                    "weight",
                    "text",
                    "coherency_analysis",
                ]
            ]
            # ].tail(20)
        )

## Tables

In [ ]:
from evaluations.analysis_utils import generate_overall_results_table

ROUND_DIGITS = 2
results_table = generate_overall_results_table(
    df_pass_at_1,
    df_samples,
    class_weights=class_weights,
    # class_weights={task_type: 1.0 for task_type in df_pass_at_1["task_type"].unique()},
    N_VALUES=N_VALUES,
    round_digits=ROUND_DIGITS,
    # include_error=True,
    include_error=False,
    include_method_info=True,
    # transpose_table=True, # for appendix
    transpose_table=False,
)

# if SAVE_FIGS:
if True:
    latex_table = results_table.to_latex(
        index=True,
        escape=True,
        float_format=f"%.{ROUND_DIGITS}f",
        multicolumn_format="c",
        multirow=False,
    )

    # MANUAL post-processing:
    # 1. Combine the index and columns into a single row
    # 2. Add \midrule between methods
    # 3. Add \textsc to the headers

    # fname = os.path.join(FIGS_PATH, "overall_results.tex")
    fname = os.path.join(FIGS_PATH, "appendix_results.tex")
    with open(fname, "w") as f:
        f.write(latex_table)

results_table

In [ ]:
# Filter to only the DisCIPL and Follower-only methods
methods_to_keep = ["DisCIPL", "Follower-only"]
# methods_to_keep = [
#     "Planner-only",
# ]
filtered_results = results_table.loc[
    results_table.index.get_level_values("Method").isin(methods_to_keep)
]["Overall"]

print(filtered_results.reset_index().to_markdown(index=False, floatfmt=f".{ROUND_DIGITS}f"))

filtered_results

# Additional analysis

In [ ]:
df_pass_at_k = compute_pass_at_k(df, k_values=N_VALUES, n_values=N_VALUES)
df_pass_at_1 = df_pass_at_k[df_pass_at_k["k"] == 1]
df_pass_at_nmax = df_pass_at_1[df_pass_at_1["N"] == df_pass_at_1["N"].max()]

In [ ]:
df_pass_at_nmax

In [ ]:
from analysis_utils import get_model_parameter_count

model_parameter_counts = {
    model: get_model_parameter_count(model) for model in df["hf_model"].unique()
}
print(model_parameter_counts)

df_pass_at_nmax.loc[:, "model_parameters"] = df_pass_at_nmax["hf_model"].map(model_parameter_counts)

In [ ]:
df_pass_at_nmax

## Parameter Scaling Summary Table

The following table shows the performance scaling from 1B, 3B, and 8B parameter models across different methods.

In [ ]:
# Create a summary table for parameter scaling
def create_parameter_scaling_table(df_pass_at_nmax):
    """
    Create a summary table showing performance scaling across 1B, 3B, 8B parameters.
    Uses class weights to properly weight performance across task types.
    """
    # Make a copy to avoid modifying original
    df = df_pass_at_nmax.copy()

    # Map model parameters to readable labels
    param_mapping = {
        1235814400: '1B',  # Llama-3.2-1B
        3212749824: '3B',  # Llama-3.2-3B
        8030261248: '8B'   # Llama-3.1-8B
    }

    # Filter to only include rows with parameter counts
    df = df[df['model_parameters'].notna()]
    df['param_size'] = df['model_parameters'].map(param_mapping)

    # Focus on key methods for scaling analysis
    key_methods = [
        'disciple_smc',
        'disciple_importance',
        'disciple_rejection',
        'vllm',
        'vllm_beam_search'
    ]

    df_filtered = df[df['pipeline_name'].isin(key_methods)]

    # Calculate weighted averages using class weights
    def weighted_mean(group):
        return (group['pass@k'] * group['class_weight']).sum() / group['class_weight'].sum()

    # Group by pipeline_name and param_size, then compute weighted average
    summary_data = []
    for pipeline in key_methods:
        row = {'pipeline_name': pipeline}
        for param_size in ['1B', '3B', '8B']:
            subset = df_filtered[
                (df_filtered['pipeline_name'] == pipeline) &
                (df_filtered['param_size'] == param_size)
            ]
            if len(subset) > 0:
                weighted_avg = weighted_mean(subset)
                row[param_size] = weighted_avg
            else:
                row[param_size] = None
        summary_data.append(row)

    # Convert to DataFrame and set index
    summary_table = pd.DataFrame(summary_data)
    summary_table.set_index('pipeline_name', inplace=True)

    # Reorder columns to 1B, 3B, 8B
    column_order = ['1B', '3B', '8B']
    summary_table = summary_table.reindex(columns=column_order)

    # Create readable method names
    method_names = {
        'disciple_smc': 'DisCIPL-SMC',
        'disciple_importance': 'DisCIPL-IS',
        'disciple_rejection': 'DisCIPL-RS',
        'vllm': 'Sampling',
        'vllm_beam_search': 'Beam Search'
    }

    summary_table.index = summary_table.index.map(method_names)

    # Round to 3 decimal places
    summary_table = summary_table.round(3)

    return summary_table

# Generate the summary table
scaling_table = create_parameter_scaling_table(df_pass_at_nmax)

# Output as Markdown with preserved trailing zeroes
print("## Parameter Scaling Summary Table (Pass@1)\n")

print(scaling_table.to_markdown())

# Also display the table for interactive viewing
scaling_table

In [ ]:
from evaluations.analysis_utils import generate_overall_results_table

ROUND_DIGITS = 2
results_table = generate_overall_results_table(
    df_pass_at_1,
    df_samples,
    class_weights=class_weights,
    N_VALUES=N_VALUES,
    round_digits=ROUND_DIGITS,
    include_error=False,
    include_method_info=True,
    transpose_table=False,
)

# Filter to only the DisCIPL and Follower-only methods
methods_to_keep = ["DisCIPL", "Follower-only"]
filtered_results = results_table.loc[
    results_table.index.get_level_values("Method").isin(methods_to_keep)
]["Overall"]

filtered_results

In [ ]:
print(filtered_results.reset_index().to_markdown(index=False, floatfmt=f".{ROUND_DIGITS}f"))

In [ ]:
sns.barplot(
    x="pipeline_name",
    y="pass@k",
    weights="class_weight",
    data=df_pass_at_nmax,
    hue="model_parameters",
    palette=None,
    capsize=0.1,
    err_kws={"linewidth": 1.5},
)

plt.xticks(rotation=90)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
pipelines = {
    "disciple_smc": "DisCIPL-SMC",
    "disciple_importance": "DisCIPL-IS",
    "disciple_rejection": "DisCIPL-RS",
    "vllm": "Sampling",
    "vllm_beam_search": "Beam Search",
}

model_name_base = "llama"

sns.lineplot(
    data=df_pass_at_nmax[
        df_pass_at_nmax["pipeline_name"].isin(pipelines.keys())
        & df_pass_at_nmax["hf_model"].str.contains(model_name_base)
    ],
    x="model_parameters",
    y="pass@k",
    weights="class_weight",
    hue="pipeline_name",
    palette=PIPELINE_PALETTE,
    marker=".",
)

plt.xscale("log")
plt.xlim(1e9, 1e10)
plt.xticks([1e9, 3e9, 8e9], ["1B", "3B", "8B"])

plt.ylim(0, 1)

plt.xlabel("Follower Model Parameters\n(Llama-3.x-Instruct)")
plt.ylabel("Pass@1")

sns.despine()

# Map legend labels to pipeline display names
handles, labels = plt.gca().get_legend_handles_labels()
mapped_labels = [pipelines.get(label, label) for label in labels]
plt.legend(
    handles, mapped_labels, bbox_to_anchor=(1.05, 1), loc="upper left", title="Method"
)

if SAVE_FIGS:
    plt.savefig(
        os.path.join(FIGS_PATH, "llama_scaling.pdf"),
        bbox_inches="tight",
        dpi=300,
    )

In [ ]:
filtered_results.reset_index()

In [ ]:
df_pareto = generate_overall_results_table(
    df_pass_at_1,
    df_samples,
    class_weights=class_weights,
    N_VALUES=N_VALUES,
    round_digits=ROUND_DIGITS,
    include_error=False,
    include_method_info=True,
    transpose_table=False,
)

df_pareto = df_pareto["Overall"].reset_index()

# df_pareto["method"] = df_pareto["index"].str.split(" (")[0]
# df_pareto["hf_model"] = df_pareto["index"].str.split(" (").str[1].str.replace(")", "")

df_pareto

In [ ]:
df_pareto = filtered_results.copy().reset_index()

df_pareto = df_pareto[
    (df_pareto["Model"].str.contains("Llama"))
]

df_pareto["Parameters"] = df_pareto["Model"].str.extract(r'(\d+)(?:B|b)')[0].astype(int)

sns.scatterplot(
    data=df_pareto,
    x="Pass@1",
    y="Coherency",
    # weights="class_weight",
    hue="Sampling Method",
    # style="Method",
    # palette=PIPELINE_PALETTE,
    size="Parameters",
)

# draw dashed lines connecting the 1B → 3B → 8B points for each method
sns.lineplot(
    data=df_pareto.sort_values("Parameters"),
    x="Pass@1",
    y="Coherency",
    hue="Sampling Method",
    # style="Method",
    dashes=True,
    markers=False,
    legend=False,
    linewidth=1
)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

if SAVE_FIGS or True:
    plt.savefig(
        os.path.join(FIGS_PATH, "pareto_plot.pdf"),
        bbox_inches="tight",
        dpi=300,
    )

In [ ]:
df_pareto

In [ ]:
df_llama_qwen = generate_overall_results_table(
    df_pass_at_1,
    df_samples,
    class_weights=class_weights,
    N_VALUES=N_VALUES,
    round_digits=ROUND_DIGITS,
    include_error=False,
    include_method_info=True,
    transpose_table=False,
)

methods_to_keep = ["DisCIPL", "Follower-only"]
df_llama_qwen = df_llama_qwen.loc[
    df_llama_qwen.index.get_level_values("Method").isin(methods_to_keep)
]["Overall"].reset_index()

df_llama_qwen = df_llama_qwen[(df_llama_qwen["Model"].isin(["Llama-3.2-1B", "Qwen3-1.7B"]))]

df_llama_qwen

In [ ]:
sns.barplot(
    data=df_llama_qwen,
    x="Model",
    y="Pass@1",
    hue="Sampling Method",
)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
df_pass_at_1.hf_model.unique()

In [ ]:
df_llama_qwen_pass_at_1 = df_pass_at_1.copy()

# remap qwen hf pipelines to vllm pipelines
df_llama_qwen_pass_at_1["pipeline_name"] = df_llama_qwen_pass_at_1[
    "pipeline_name"
].replace({"huggingface": "vllm", "huggingface_beam_search": "vllm_beam_search"})


df_llama_qwen_pass_at_1 = df_llama_qwen_pass_at_1[
    df_llama_qwen_pass_at_1["pipeline_name"].isin(
        [
            "disciple_smc",
            "disciple_importance",
            "disciple_rejection",
            "vllm",
            "vllm_beam_search",
            "huggingface",
            "huggingface_beam_search",
        ]
    )
    & df_llama_qwen_pass_at_1["hf_model"].isin(
        ["meta-llama/Llama-3.2-1B-Instruct", "Qwen/Qwen3-1.7B"]
    )
]

from analysis_utils import get_pipeline_info

df_llama_qwen_pass_at_1["sampling_method"] = df_llama_qwen_pass_at_1.apply(
    lambda row: get_pipeline_info(row["pipeline_name"], row["hf_model"])[
        "sampling_method"
    ],
    axis=1,
)

df_llama_qwen_pass_at_1["hf_model_short"] = df_llama_qwen_pass_at_1.apply(
    lambda row: get_pipeline_info(row["pipeline_name"], row["hf_model"])[
        "model_short"
    ],
    axis=1,
)

df_llama_qwen_pass_at_1["method_short"] = df_llama_qwen_pass_at_1.apply(
    lambda row: get_pipeline_info(row["pipeline_name"], row["hf_model"])[
        "method_short"
    ],
    axis=1,
)

ax_bar = sns.barplot(
    x="method_short",
    y="pass@k",
    weights="class_weight",
    data=df_llama_qwen_pass_at_1,
    hue="hf_model_short",
    palette="Paired",
    capsize=0.1,
    err_kws={"linewidth": 1.5},
)

ax_bar.set_xlabel("")
ax_bar.set_ylabel("Pass@1", fontsize=14)
ax_bar.tick_params(axis="both", which="major", labelsize=12, length=6)
sns.despine(ax=ax_bar, top=True, right=True)
ax_bar.get_legend().remove()
ax_bar.set_ylim(0, 1)  # Ensure y-axis is [0, 1]

# plt.legend(bbox_to_anchor=(0.5, 1.0), loc='upper center', ncol=2, fontsize=12)

if SAVE_FIGS or True:
    plt.savefig(
        os.path.join(FIGS_PATH, "llama_vs_qwen_nolegend.pdf"), bbox_inches="tight", dpi=300
    )

## Compute cost

In [ ]:
from evaluations.analysis_utils import calculate_disciple_token_usage

# This takes about 30-60s since it requires reading the completion.json files
calculate_disciple_token_usage(df)

In [ ]:
methods = [
    "DisCIPL-SMC (Llama-3.2-1B)",
    "Llama-3.2-1B",
    "GPT-4o",
    "GPT-4o (CoT)",
    "o1",
]

token_df = df[df["method"].isin(methods)].copy()
token_df["method"] = pd.Categorical(
    token_df["method"],
    categories=methods,
    ordered=True,
)

# Compute answer tokens for all methods
# For 'gpt-4o-cot', subtract reasoning_tokens from completion_tokens
token_df["answer_tokens"] = df.apply(
    lambda row: (
        row["completion_tokens"] - row["reasoning_tokens"]
        if row["pipeline_name"] == "gpt-4o-cot"
        else row["completion_tokens"]
    ),
    axis=1,
)

In [ ]:
TOKEN_COLS = {
    "answer_tokens": "Answer",
    "reasoning_tokens": "Reasoning",
    "prompt_tokens": "Task/Follower Prompt",
    "model_generator_prompt_tokens": "Planner Prompt",
    "model_generator_prompt_tokens_cached": "Planner Prompt (Cached)",
}

token_summary = (
    token_df.groupby(["method"], observed=True)[list(TOKEN_COLS.keys())]
    .agg(["mean", "std"])
    .rename(columns=TOKEN_COLS, level=0)
    .round(1)
)

# Clean up token_summary for markdown output
flat_entries = []
for _, row in token_summary.reset_index().iterrows():
    entry = {"Method": row['method'].item()}
    # for each token‐column, combine mean and std or put dash if NaN
    for col_name in token_summary.columns.levels[0]:
        mean = row[(col_name, 'mean')]
        std = row[(col_name, 'std')]
        entry[col_name] = '-' if pd.isna(mean) else f"{mean:.1f} ({std:.1f})"
    flat_entries.append(entry)
flat_df = pd.DataFrame(flat_entries)

flat_df = flat_df[["Method"] + list(TOKEN_COLS.values())]

print(flat_df.to_markdown(index=False, floatfmt=".1f"))

flat_df

In [ ]:
# only display these methods in the plot
methods = [
    "o1",
    "GPT-4o (CoT)",
    "DisCIPL-SMC (Llama-3.2-1B)",
]
_df = token_df.copy()
_df = _df[_df["method"].isin(methods)]
_df["method"] = pd.Categorical(
    _df["method"],
    categories=methods,
    ordered=True,
)

sns.barplot(
    data=_df,
    x="task_type",
    y="reasoning_tokens",
    hue="method",
    palette=PALETTE,
    capsize=0.1,
    err_kws={"linewidth": 1.5},
)
sns.despine()
plt.legend(title="")
plt.ylabel("Token Usage\n(Reasoning / Program)")
plt.xlabel(f"{DATASET.upper()} {SUBTASKS}")

if SAVE_FIGS:
    plt.savefig(
        os.path.join(FIGS_PATH, "token_usage_reasoning.pdf"),
        bbox_inches="tight",
        dpi=300,
    )

## Cost Analysis

Now we'll calculate the dollar costs for each method based on token counts and the pricing information.

In [ ]:
# Cost per 1M tokens (in USD)
COST_RATES = {
    'llama_input': 0.06,    # Llama-3.2-1B input
    'llama_output': 0.06,   # Llama-3.2-1B output
    'gpt4o_input': 5.00,    # GPT-4o input
    'gpt4o_cached': 2.50,   # GPT-4o cached input
    'gpt4o_output': 20.00,  # GPT-4o output
    'o1_input': 15.00,      # o1 input
    'o1_cached': 7.50,      # o1 cached input
    'o1_output': 60.00      # o1 output
}

def calculate_cost_for_method(row, method):
    """Calculate cost for a single method based on token counts."""

    if method == "DisCIPL-SMC (Llama-3.2-1B)":
        # DisCIPL uses both GPT-4o (Planner) and Llama (Follower)

        # Follower costs (Llama-3.2-1B)
        follower_prompt_cost = (row['prompt_tokens'] / 1e6) * COST_RATES['llama_input']
        follower_output_cost = (row['answer_tokens'] / 1e6) * COST_RATES['llama_output']

        # Planner costs (GPT-4o)
        # Non-cached tokens = total - cached
        planner_noncached_tokens = row['model_generator_prompt_tokens'] - row['model_generator_prompt_tokens_cached']
        planner_noncached_cost = (planner_noncached_tokens / 1e6) * COST_RATES['gpt4o_input']
        planner_cached_cost = (row['model_generator_prompt_tokens_cached'] / 1e6) * COST_RATES['gpt4o_cached']
        planner_output_cost = (row['reasoning_tokens'] / 1e6) * COST_RATES['gpt4o_output']
        total_cost = (
            follower_output_cost
            + planner_output_cost
            + follower_prompt_cost
            + planner_noncached_cost
            + planner_cached_cost
        )

        return {
            "answer_cost": follower_output_cost,
            "reasoning_cost": planner_output_cost,
            "follower_prompt_cost": follower_prompt_cost,
            "planner_noncached_cost": planner_noncached_cost,
            "planner_cached_cost": planner_cached_cost,
            "total_cost": total_cost,
        }

    elif method == "Llama-3.2-1B":
        # Only Llama costs
        input_cost = (row['prompt_tokens'] / 1e6) * COST_RATES['llama_input']
        output_cost = (row['answer_tokens'] / 1e6) * COST_RATES['llama_output']
        total_cost = input_cost + output_cost

        return {
            "answer_cost": output_cost,
            "reasoning_cost": 0.0,
            "follower_prompt_cost": input_cost,
            "planner_noncached_cost": 0.0,
            "planner_cached_cost": 0.0,
            "total_cost": total_cost
        }

    elif method == "GPT-4o":
        # Only GPT-4o costs
        input_cost = (row['prompt_tokens'] / 1e6) * COST_RATES['gpt4o_input']
        output_cost = (row['answer_tokens'] / 1e6) * COST_RATES['gpt4o_output']
        total_cost = input_cost + output_cost

        return {
            "answer_cost": output_cost,
            "reasoning_cost": 0.0,
            "follower_prompt_cost": input_cost,
            "planner_noncached_cost": 0.0,
            "planner_cached_cost": 0.0,
            "total_cost": total_cost
        }

    elif method == "GPT-4o (CoT)":
        # GPT-4o with reasoning
        input_cost = (row['prompt_tokens'] / 1e6) * COST_RATES['gpt4o_input']
        reasoning_cost = (row['reasoning_tokens'] / 1e6) * COST_RATES['gpt4o_output']
        output_cost = (row['answer_tokens'] / 1e6) * COST_RATES['gpt4o_output']
        total_cost = input_cost + reasoning_cost + output_cost
        return {
            "answer_cost": output_cost,
            "reasoning_cost": reasoning_cost,
            "follower_prompt_cost": input_cost,
            "planner_noncached_cost": 0.0,
            "planner_cached_cost": 0.0,
            "total_cost": total_cost
        }

    elif method == "o1":
        # o1 costs
        input_cost = (row['prompt_tokens'] / 1e6) * COST_RATES['o1_input']
        reasoning_cost = (row['reasoning_tokens'] / 1e6) * COST_RATES['o1_output']
        output_cost = (row['answer_tokens'] / 1e6) * COST_RATES['o1_output']
        total_cost = input_cost + reasoning_cost + output_cost
        return {
            "answer_cost": output_cost,
            "reasoning_cost": reasoning_cost,
            "follower_prompt_cost": input_cost,
            "planner_noncached_cost": 0.0,
            "planner_cached_cost": 0.0,
            "total_cost": total_cost
        }

    else:
        raise ValueError(f"Unknown method: {method}")

# Calculate costs for each row
cost_data = []
for method in methods:
    subset = token_df[token_df['method'] == method]
    for _, row in subset.iterrows():
        cost_info = calculate_cost_for_method(row, method)
        cost_info['method'] = method
        # cost_info['task_type'] = row['task_type']
        # cost_info['task_id'] = row['task_id']
        cost_data.append(cost_info)

# Create a DataFrame for the cost data

cost_df = pd.DataFrame(cost_data)

#
# .groupby(["method"]).mean()
# print("Cost Analysis (per task):")
# print(cost_df.to_markdown(index=False))
# print()
# cost_df

In [ ]:
TOKEN_COLS

In [ ]:
COST_COLS = {
    "answer_cost": "Answer",
    "reasoning_cost": "Reasoning",
    "follower_prompt_cost": "Follower Prompt",
    "planner_noncached_cost": "Planner Prompt (Non-Cached)",
    "planner_cached_cost": "Planner Prompt (Cached)",
    "total_cost": "Total Cost"
}

# Group by method and calculate the mean
cost_summary = cost_df.groupby(["method"]).mean()

# Rename the columns using COST_COLS
cost_summary.rename(columns=COST_COLS, inplace=True)

cost_summary.index.name = ""

# Add dollar signs to each value
cost_summary = cost_summary * 100
cost_summary = cost_summary.applymap(lambda x: f"${x:.4f}")

# Convert to markdown
markdown_output = cost_summary.to_markdown()
print(markdown_output)

cost_summary

In [ ]:
cost_summary.index.name

## Cost Calculation Methodology

The cost analysis above uses the following approach:

### Pricing (per 1M tokens)
- **Llama-3.2-1B**: $0.06 input, $0.06 output
- **GPT-4o**: $5.00 input, $2.50 cached input, $20.00 output  
- **o1**: $15.00 input, $7.50 cached input, $60.00 output

### Method-specific calculations

1. **DisCIPL-SMC**: Uses both GPT-4o (Planner) and Llama-3.2-1B (Follower)
   - Follower: `prompt_tokens` × Llama input rate + `answer_tokens` × Llama output rate
   - Planner: Non-cached tokens × GPT-4o input rate + Cached tokens × GPT-4o cached rate + `reasoning_tokens` × GPT-4o output rate
   - Non-cached tokens = `model_generator_prompt_tokens` - `model_generator_prompt_tokens_cached`

2. **GPT-4o**: Standard GPT-4o costs
   - `prompt_tokens` × GPT-4o input rate + `answer_tokens` × GPT-4o output rate

3. **GPT-4o (CoT)**: GPT-4o with chain-of-thought reasoning
   - `prompt_tokens` × GPT-4o input rate + (`reasoning_tokens` + `answer_tokens`) × GPT-4o output rate

4. **o1**: Uses o1 pricing
   - `prompt_tokens` × o1 input rate + (`reasoning_tokens` + `answer_tokens`) × o1 output rate

5. **Llama-3.2-1B**: Only Llama costs (but lacks complete token data in this dataset)

### Notes
- For DisCIPL, cached tokens represent the portion of the planner prompt that is cached across multiple tasks
- All costs are calculated per individual task
- Missing token data is handled by excluding those samples from the analysis